In [15]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [16]:
column_names = (
    ["unit", "cycle"]
    + ["setting_1", "setting_2", "setting_3"]
    + [f"sensor_{i}" for i in range(1, 22)]
)

train_df = pd.read_csv(
    "../data/raw/train_FD001.txt",
    sep=r"\s+",
    header=None,
    names=column_names
)
train_df.head()
max_cycles = train_df.groupby("unit")["cycle"].transform("max")

train_df["RUL"] = max_cycles - train_df["cycle"]

In [17]:
sensor_columns = [f"sensor_{i}" for i in range(1, 22)]
sensor_std = train_df[sensor_columns].std()
sensor_std.sort_values()
low_variance_sensors = sensor_std[sensor_std < 0.01]
selected_sensors = [
    sensor for sensor in sensor_columns
    if sensor not in low_variance_sensors.index
]


In [18]:
feature_columns = selected_sensors
X = train_df[feature_columns]
y = train_df["RUL"]
units = train_df["unit"].unique()
train_units, val_units = train_test_split(
    units,
    test_size=0.2,
    random_state=42
)

In [19]:
train_data = train_df[
    train_df["unit"].isin(train_units)
].copy()

val_data = train_df[
    train_df["unit"].isin(val_units)
].copy()
set(train_units).intersection(set(val_units))

set()

In [20]:
X_train = train_data[feature_columns]
y_train = train_data["RUL"]

X_val = val_data[feature_columns]
y_val = val_data["RUL"]
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

X_train: (16561, 14)
y_train: (16561,)
X_val: (4070, 14)
y_val: (4070,)


In [21]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=feature_columns,
    index=X_train.index
)

X_val_scaled = pd.DataFrame(
    X_val_scaled,
    columns=feature_columns,
    index=X_val.index
)
X_train_scaled.head()

,sensor_2,sensor_3,sensor_4,sensor_7,sensor_8,sensor_9,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_17,sensor_20,sensor_21
192,-1.581191,-1.092662,-1.963014,1.318090,-1.221017,-0.471297,-2.294365,1.242127,-0.500147,-0.315944,-1.373754,-1.426246,0.678684,1.564436
193,-1.721722,-0.565155,-1.756664,1.591518,-1.645675,-0.621789,-1.127747,1.745551,-1.615806,-0.654442,-0.675217,-0.779529,1.345844,1.098895
194,-2.263772,-0.356453,-1.106382,2.013053,-0.796358,-0.492481,-1.203012,1.582278,-1.476349,-0.169925,-1.648369,-1.426246,1.623827,1.252524
195,-2.002785,-1.041719,-1.427619,0.999091,-1.645675,-0.326102,-1.654606,1.459824,-2.313093,-0.177072,-1.072476,-1.426246,1.735021,1.975974
196,-1.902406,-1.883102,-0.709298,1.990267,-0.937911,-0.319923,-1.090114,1.160491,-2.173636,-0.369552,-1.475068,-2.072963,2.013004,1.237627


In [22]:
train_processed = X_train_scaled.copy()

train_processed["unit"] = train_data["unit"].values
train_processed["cycle"] = train_data["cycle"].values
train_processed["RUL"] = y_train.values
val_processed = X_val_scaled.copy()

val_processed["unit"] = val_data["unit"].values
val_processed["cycle"] = val_data["cycle"].values
val_processed["RUL"] = y_val.values

In [23]:
train_processed.to_csv(
    "../data/processed/train_processed.csv",
    index=False
)

val_processed.to_csv(
    "../data/processed/validation_processed.csv",
    index=False
)

In [24]:
joblib.dump(
    scaler,
    "../models/scaler.joblib"
)

['../models/scaler.joblib']